# 13 — Penrose SVG and TikZ renderers (v1.5 patch)

**Spacetime Lab v1.5 — pure-string SVG and TikZ backends for Penrose diagrams**

Phase 2 (v0.2.0) shipped the matplotlib backend for Penrose diagrams. That works perfectly for Jupyter notebooks but is the wrong tool for two important targets:

- **Web frontends** — matplotlib's PNG output is rasterised and bulky; SVG is vector, lightweight, and stylable with CSS.
- **LaTeX papers** — embedding matplotlib figures means PDF rasterisation drift; TikZ is text, version-controllable, and compiles natively at the document's font size.

v1.5 ships pure-string `render_svg` and `render_tikz` functions. Both consume the same `Scene` data structure as `render_matplotlib` and share a kind-to-style table so the three backends produce visually consistent diagrams.


In [ ]:
from IPython.display import SVG, display, HTML

from spacetime_lab.diagrams import (
    MinkowskiPenrose,
    SchwarzschildPenrose,
)
from spacetime_lab.diagrams.render import (
    render_svg,
    render_tikz,
    render_matplotlib,
)

minkowski = MinkowskiPenrose().to_scene()
schwarzschild = SchwarzschildPenrose(mass=1.0).to_scene()
print(f'Minkowski:     {len(minkowski.paths)} paths, {len(minkowski.infinities)} infinities')
print(f'Schwarzschild: {len(schwarzschild.paths)} paths, {len(schwarzschild.infinities)} infinities')

## 1. SVG render — Minkowski

The diamond with infinities labelled in the document font. Paths are grouped into `<g class="kind-{kind}">` so a downstream stylesheet can theme entire categories without parsing individual `<path>` attributes.

In [ ]:
svg_minkowski = render_svg(minkowski, width=380, height=380)
display(SVG(svg_minkowski))
print(f'SVG size: {len(svg_minkowski)} bytes')

## 2. SVG render — Schwarzschild (eternal four-region)

Same data, much richer geometry: four causal regions, two horizons, two singularities, two infinities per region.

In [ ]:
svg_schwarzschild = render_svg(schwarzschild, width=480, height=480)
display(SVG(svg_schwarzschild))
print(f'SVG size: {len(svg_schwarzschild)} bytes')

## 3. CSS theming demonstration

Because every path is in a `kind-*` group, a single CSS rule can re-skin all horizons or all singularities without touching the SVG source. Below: a high-contrast 'paper white' palette and a low-contrast 'dark mode' palette.

In [ ]:
themed = f'''
<div style="display: flex; gap: 24px;">
  <div>
    <h4>paper-white theme</h4>
    <style>
      .paper-white .kind-boundary path {{ stroke: #000; stroke-width: 2.4; }}
      .paper-white .kind-horizon path  {{ stroke: #c00; stroke-dasharray: 6 4; }}
      .paper-white .kind-singularity path {{ stroke: #c00; stroke-dasharray: 2 2; }}
    </style>
    <div class="paper-white">{svg_schwarzschild}</div>
  </div>
  <div>
    <h4>dark theme</h4>
    <style>
      .dark {{ background: #111; padding: 8px; border-radius: 6px; }}
      .dark rect[fill="white"] {{ fill: #111 !important; }}
      .dark .kind-boundary path {{ stroke: #eee; }}
      .dark .kind-horizon path  {{ stroke: #5cf; }}
      .dark .kind-singularity path {{ stroke: #f53; }}
      .dark text {{ fill: #eee; }}
    </style>
    <div class="dark">{svg_schwarzschild}</div>
  </div>
</div>
'''
display(HTML(themed))

## 4. TikZ output — Minkowski

Pure-string LaTeX source. Drop into a `.tex` file with `\usepackage{tikz}` and `\usepackage{mathrsfs}` to compile.

In [ ]:
tikz_m = render_tikz(minkowski, scale=2.5)
print(tikz_m)

## 5. TikZ standalone document — Schwarzschild

Setting `standalone=True` wraps the picture in a complete document preamble, ready to compile with `pdflatex`:

In [ ]:
tikz_s = render_tikz(schwarzschild, scale=3.0, standalone=True)
# Show the first chunk so the cell output stays readable
lines = tikz_s.splitlines()
print('\n'.join(lines[:18]))
print(f'\n... ({len(lines)} lines total) ...\n')
print('\n'.join(lines[-5:]))

## 6. Three-backend cross-check — same Scene, three outputs

All three backends consume the same `Scene` and produce visually consistent diagrams. The matplotlib backend below is for comparison; the SVG and TikZ outputs above use the same shared kind-to-style table.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
render_matplotlib(minkowski, ax=axes[0])
render_matplotlib(schwarzschild, ax=axes[1])
plt.tight_layout()
plt.show()

## 7. Closing gate — structural invariants

In [ ]:
import xml.etree.ElementTree as ET

for name, scene in [('Minkowski', minkowski), ('Schwarzschild', schwarzschild)]:
    svg = render_svg(scene)
    tz = render_tikz(scene)
    # SVG must be well-formed XML
    ET.fromstring(svg)
    # Both backends emit one drawable per Path and one label per Infinity
    assert svg.count('<path ') == len(scene.paths)
    assert tz.count('\\draw[') == len(scene.paths)
    assert svg.count('<text ') == len(scene.infinities)
    assert tz.count('\\node ') == len(scene.infinities)
    # Every kind in scene appears as a group/section in both backends
    for kind in {p.kind for p in scene.paths}:
        assert f'class="kind-{kind}"' in svg
        assert f'% --- {kind} ---' in tz
    print(f'{name}: SVG {len(svg)} bytes, TikZ {len(tz)} chars — all gates pass')

print()
print('ALL GATES PASS — v1.5 SVG + TikZ renderers are good to ship.')

## Scope notes

**v1.5 ships:**
- `render_svg(scene, width, height, padding, show_labels, label_offset)` — pure-string SVG, no runtime deps, well-formed XML, kind-grouped for CSS theming
- `render_tikz(scene, scale, show_labels, label_offset, standalone)` — pure-string LaTeX/TikZ source, optional standalone document preamble
- Shared kind-to-style mapping with `render_matplotlib` (boundary solid black, horizon dashed grey, singularity short-dashed red, etc.)
- 29 new tests pinned to structural invariants (envelope shape, well-formedness, kind grouping, infinity count, dasharray, override behaviour)

**Not in scope (deferred or out of scope):**
- **PNG / JPEG / PDF rasterisers** — these belong downstream (browser SVG renderer, `rsvg-convert`, `pdflatex`)
- **Interactive SVG** — clicks, hovers, animation hooks; that's a frontend concern
- **Full LaTeX glyph fidelity in SVG** — SVG cannot render arbitrary LaTeX; we use Unicode `ℐ` for `\mathscr{I}`, which renders well in any modern browser font
- **Embedded Penrose backend** (e.g. https://penrose.cs.cmu.edu) — different ecosystem, different goals

The two new renderers complete the Phase 2 'three backends' design from v0.2.0. Everything in the diagrams subsystem is now non-stub and tested.